# 04. Clustering de trayectorias

Agrupa a los artistas por la forma de sus trayectorias, con tres distancias entre series y tres algoritmos de clustering. Se construye por etapas:

1. **Matrices de distancia** (esta versión): euclídea, DTW y SBD.
2. Clustering: jerárquico, spectral y PAM sobre cada matriz.
3. Elección de k.
4. Estabilidad con semillas.
5. Control por decil de popularidad.

**Entrada:** panel preprocesado del notebook 03 (`valor_suavizado`: log1p centrado y media móvil de 3 meses).
**Referencia:** Paparrizos et al., *A Survey on Time-Series Distance Measures*, [arXiv:2412.20574](https://arxiv.org/abs/2412.20574) (2024).

In [ ]:
# PAQUETES

library(data.table)
library(dtwclust)
library(ggplot2)

# Tipografias de la presentacion (se dibujan con ragg): Berlin Sans FB Demi
# para titulos y DejaVu Sans Mono para el resto (copia en fuentes/)
systemfonts::register_font("LWR Titulo", plain = "C:/Windows/Fonts/BRLNSDB.TTF")
systemfonts::register_font(
  "LWR Mono",
  plain = "fuentes/DejaVuSansMono.ttf",
  bold = "fuentes/DejaVuSansMono-Bold.ttf"
)

# dtwclust cambia el generador aleatorio al cargarse: se fija el
# generador por defecto de R para que las semillas sean reproducibles
RNGkind("Mersenne-Twister", "Inversion", "Rejection")

## 1. Configuración y carga

In [ ]:
# ============================================================
# CONFIGURACION
# ============================================================

# Panel preprocesado del notebook 03: el más reciente del directorio
archivos_panel_preprocesado <- list.files(
  pattern = "^artistas_ST_[0-9]+de[0-9]+_top[0-9]+_22_26_preprocesado_.+_long[.]csv$"
)

stopifnot(
  "No hay panel preprocesado: correr el notebook 03" =
    length(archivos_panel_preprocesado) > 0
)

ARCHIVO_PANEL_PREPROCESADO <- archivos_panel_preprocesado[
  which.max(file.mtime(archivos_panel_preprocesado))
]

VARIABLES_CLUSTERING <- c(
  "spotify_followers",
  "instagram_followers",
  "spotify_playlist_count",
  "youtube_channel_subscribers",
  "youtube_playlist_count"
)

# Serie de entrada de las distancias y serie de la prueba de
# sensibilidad sin suavizar
COLUMNA_SERIE <- "valor_suavizado"
COLUMNA_SERIE_SIN_SUAVIZAR <- "valor_transformado"

# DTW: corrimiento máximo en meses (banda de Sakoe-Chiba). 6 meses
# es la ventana principal; 3 y 12, sensibilidad. Con ventana 0,
# DTW coincide con la euclídea.
VENTANA_DTW_MESES <- 6L
VENTANAS_DTW_SENSIBILIDAD <- c(3L, 12L)

# Patrón de pasos clásico y norma L2: con ventana 0 la distancia es
# exactamente la euclídea, así las dos quedan en la misma escala
PATRON_PASOS_DTW <- dtw::symmetric1
NORMA_DTW <- "L2"

# Semilla primigenia del workflow; las demás semillas se derivan
# de ella con generar_semillas_primas()
SEMILLA_PRIMIGENIA <- 999199L

OUT_DIR <- "chartmetric_pilotos"
ARCHIVO_DISTANCIAS <- file.path(OUT_DIR, "distancias_clustering.rds")
FIGURES_DIR <- "figuras"
dir.create(FIGURES_DIR, showWarnings = FALSE, recursive = TRUE)

ARCHIVO_PANEL_PREPROCESADO

In [ ]:
# ============================================================
# SEMILLAS: primos de 6 cifras sorteados a partir de la
# semilla primigenia. La misma n devuelve siempre las mismas
# semillas, y pedir más conserva las primeras.
# ============================================================

primos_seis_cifras <- function() {
  limite <- 999999L
  es_primo <- rep(TRUE, limite)
  es_primo[1] <- FALSE
  for (p in 2:floor(sqrt(limite))) {
    if (es_primo[p]) es_primo[seq(p * p, limite, by = p)] <- FALSE
  }
  candidatos <- which(es_primo)
  candidatos[candidatos >= 100000L]
}

PRIMOS_SEIS_CIFRAS <- primos_seis_cifras()

generar_semillas_primas <- function(n, semilla = SEMILLA_PRIMIGENIA) {
  set.seed(semilla)
  orden <- sample(length(PRIMOS_SEIS_CIFRAS))
  PRIMOS_SEIS_CIFRAS[orden[seq_len(n)]]
}

# Sanity check: reproducibles, primos de 6 cifras y estables al pedir más
semillas_control <- generar_semillas_primas(10)

stopifnot(
  identical(semillas_control, generar_semillas_primas(10)),
  identical(semillas_control[1:5], generar_semillas_primas(5)),
  all(semillas_control >= 100000 & semillas_control <= 999999),
  all(sapply(semillas_control, function(s) all(s %% 2:floor(sqrt(s)) != 0)))
)

semillas_control

In [ ]:
# ============================================================
# CARGA: una matriz de 49 meses x 5 variables por artista
# ============================================================

panel <- fread(
  ARCHIVO_PANEL_PREPROCESADO,
  select = c("artist_id", "artist_name", "cm_artist_rank", "decil_marco",
             "variable", "month", COLUMNA_SERIE, COLUMNA_SERIE_SIN_SUAVIZAR)
)

panel[, month := as.Date(month)]
setkey(panel, artist_id, month)

artistas <- unique(panel[, .(artist_id, artist_name, cm_artist_rank, decil_marco)])
setorder(artistas, artist_id)

armar_series <- function(columna) {
  ancho <- dcast(panel, artist_id + month ~ variable, value.var = columna)
  filas_por_artista <- split(seq_len(nrow(ancho)), ancho$artist_id)
  series <- lapply(filas_por_artista, function(filas) as.matrix(ancho[filas, ..VARIABLES_CLUSTERING]))
  series[as.character(artistas$artist_id)]
}

series_suavizadas <- armar_series(COLUMNA_SERIE)
series_sin_suavizar <- armar_series(COLUMNA_SERIE_SIN_SUAVIZAR)

MESES <- sort(unique(panel$month))

In [ ]:
# ------------------------------------------------------------
# SANITY CHECK DE LA CARGA: un artista por serie, 49 x 5, sin NA
# ------------------------------------------------------------

stopifnot(
  length(series_suavizadas) == nrow(artistas),
  identical(names(series_suavizadas), names(series_sin_suavizar)),
  all(vapply(series_suavizadas, function(m) identical(dim(m), c(length(MESES), length(VARIABLES_CLUSTERING))), logical(1))),
  !anyNA(unlist(series_suavizadas)),
  !anyNA(unlist(series_sin_suavizar))
)

cat(
  "Artistas:", length(series_suavizadas), "\n",
  "Meses:", length(MESES), "(", format(min(MESES)), "a", format(max(MESES)), ")\n",
  "Variables:", paste(VARIABLES_CLUSTERING, collapse = ", "), "\n",
  "Pares de artistas por matriz:", choose(length(series_suavizadas), 2), "\n"
)

## 2. Matrices de distancia

Una distancia por categoría de la survey, calculada una sola vez para todos los algoritmos:

- **Euclídea** (lock-step): compara mes contra mes.
- **DTW dependiente** (elástica): un solo alineamiento para las 5 variables, con corrimiento máximo de 6 meses. Tolera eventos corridos en el tiempo.
- **SBD** (sliding): 1 menos la correlación cruzada normalizada máxima (la ccf de la clase 05). Tolera un corrimiento global. `dtwclust` la calcula por variable; se promedian las 5 (versión independiente, sec. 11.4 de la survey).

El ejemplo sintético muestra la diferencia: la misma explosión corrida 3 meses.

In [ ]:
# ============================================================
# EJEMPLO SINTETICO: la misma explosión, corrida 3 meses
# ============================================================

meses_ejemplo <- seq_along(MESES)
explosion <- function(mes_centro) exp(-((meses_ejemplo - mes_centro) / 3)^2)

serie_a <- matrix(explosion(20))
serie_b <- matrix(explosion(23))

ejemplo_distancias <- data.table(
  ventana_meses = c(0L, 1L, 2L, 3L, 6L),
  dtw = sapply(c(0, 1, 2, 3, 6), function(w)
    dtw_basic(serie_a, serie_b, window.size = w, norm = NORMA_DTW, step.pattern = PATRON_PASOS_DTW))
)

ejemplo_distancias[, euclidea := sqrt(sum((serie_a - serie_b)^2))]

# Sanity check: con ventana 0 DTW es la euclídea; con ventana >= 3,
# el corrimiento queda absorbido por completo
stopifnot(
  isTRUE(all.equal(ejemplo_distancias[ventana_meses == 0, dtw], ejemplo_distancias$euclidea[1])),
  all(ejemplo_distancias[ventana_meses >= 3, dtw] < 1e-12)
)

ejemplo_distancias

In [ ]:
# ============================================================
# FUNCIONES DE DISTANCIA
# Devuelven matrices densas con los artist_id como nombres.
# ============================================================

distancia_euclidea <- function(series) {
  as.matrix(dist(t(vapply(series, as.vector, numeric(length(series[[1]]))))))
}

distancia_dtw <- function(series, ventana) {
  as.matrix(proxy::dist(
    series,
    method = "dtw_basic",
    window.size = ventana,
    norm = NORMA_DTW,
    step.pattern = PATRON_PASOS_DTW
  ))
}

distancia_sbd <- function(series) {
  por_variable <- lapply(seq_along(VARIABLES_CLUSTERING), function(j) {
    as.matrix(proxy::dist(lapply(series, function(m) m[, j]), method = "SBD"))
  })
  Reduce(`+`, por_variable) / length(por_variable)
}

In [ ]:
# ============================================================
# CALCULO DE LAS MATRICES
# Principales: euclídea, DTW con VENTANA_DTW_MESES y SBD.
# Sensibilidad: DTW con otras ventanas y DTW sin suavizar.
# ============================================================

tiempos_calculo <- list()

calcular <- function(nombre, expresion) {
  tiempo <- system.time(resultado <- expresion)[["elapsed"]]
  tiempos_calculo[[nombre]] <<- tiempo
  dimnames(resultado) <- list(names(series_suavizadas), names(series_suavizadas))
  resultado
}

matrices_distancia <- list(
  euclidea = calcular("euclidea", distancia_euclidea(series_suavizadas)),
  dtw = calcular("dtw", distancia_dtw(series_suavizadas, VENTANA_DTW_MESES)),
  sbd = calcular("sbd", distancia_sbd(series_suavizadas))
)

matrices_sensibilidad <- c(
  setNames(
    lapply(VENTANAS_DTW_SENSIBILIDAD, function(w)
      calcular(paste0("dtw_ventana_", w), distancia_dtw(series_suavizadas, w))),
    paste0("dtw_ventana_", VENTANAS_DTW_SENSIBILIDAD)
  ),
  list(dtw_sin_suavizar = calcular("dtw_sin_suavizar", distancia_dtw(series_sin_suavizar, VENTANA_DTW_MESES)))
)

data.table(matriz = names(tiempos_calculo), segundos = round(unlist(tiempos_calculo), 1))

In [ ]:
# ------------------------------------------------------------
# SANITY CHECK DE LAS MATRICES
# - cuadradas, simétricas, diagonal 0, sin NA ni negativos;
# - SBD entre 0 y 2;
# - DTW no aumenta al ampliar la ventana y nunca supera a la
#   euclídea (ventana 0).
# ------------------------------------------------------------

todas_las_matrices <- c(matrices_distancia, matrices_sensibilidad)
n_artistas <- length(series_suavizadas)

control_matrices <- rbindlist(lapply(names(todas_las_matrices), function(nombre) {
  m <- todas_las_matrices[[nombre]]
  data.table(
    matriz = nombre,
    dimension_ok = identical(dim(m), c(n_artistas, n_artistas)),
    simetrica = isSymmetric(unname(m), tol = 1e-8),
    diagonal_cero = max(abs(diag(m))) < 1e-8,
    sin_na_ni_negativos = !anyNA(m) && min(m) >= 0,
    mediana = signif(median(m[upper.tri(m)]), 4)
  )
}))

tolerancia <- 1e-8
orden_dtw_ok <- all(matrices_sensibilidad$dtw_ventana_12 <= matrices_distancia$dtw + tolerancia) &&
  all(matrices_distancia$dtw <= matrices_sensibilidad$dtw_ventana_3 + tolerancia) &&
  all(matrices_sensibilidad$dtw_ventana_3 <= matrices_distancia$euclidea + tolerancia)

stopifnot(
  all(control_matrices[, dimension_ok & simetrica & diagonal_cero & sin_na_ni_negativos]),
  max(matrices_distancia$sbd) <= 2,
  "DTW debería ser no creciente con la ventana y menor o igual que la euclídea" = orden_dtw_ok
)

control_matrices

## 3. Comparación entre distancias

Antes de agrupar: cuánto se parecen las distancias entre sí y qué pares de artistas cambian más entre la euclídea y DTW. Correlación de Spearman entre matrices (orden de los pares) y vecinos compartidos (de los 10 artistas más cercanos a cada uno, cuántos coinciden).

In [ ]:
# ============================================================
# ACUERDO ENTRE MATRICES
# ============================================================

triangulo <- function(m) m[upper.tri(m)]

correlacion_spearman <- function(m1, m2) cor(triangulo(m1), triangulo(m2), method = "spearman")

vecinos_compartidos <- function(m1, m2, k = 10) {
  vecinos <- function(m) apply(m, 1, function(fila) order(fila)[2:(k + 1)])
  v1 <- vecinos(m1)
  v2 <- vecinos(m2)
  median(vapply(seq_len(ncol(v1)), function(i) length(intersect(v1[, i], v2[, i])), integer(1)))
}

pares_comparacion <- data.table(
  matriz_1 = c("euclidea", "euclidea", "dtw", "dtw", "dtw", "dtw"),
  matriz_2 = c("dtw", "sbd", "sbd", "dtw_ventana_3", "dtw_ventana_12", "dtw_sin_suavizar")
)

acuerdo_matrices <- pares_comparacion[
  ,
  .(
    spearman = round(correlacion_spearman(todas_las_matrices[[matriz_1]], todas_las_matrices[[matriz_2]]), 3),
    vecinos_compartidos_de_10 = vecinos_compartidos(todas_las_matrices[[matriz_1]], todas_las_matrices[[matriz_2]])
  ),
  by = .(matriz_1, matriz_2)
]

acuerdo_matrices

In [ ]:
# ============================================================
# PARES QUE DTW ACERCA MAS RESPECTO DE LA EUCLIDEA
# Cociente DTW / euclídea: cerca de 0, la diferencia entre los
# dos artistas era sobre todo un corrimiento en el tiempo.
# Candidatos para la figura que explica el método.
# ============================================================

indices_triangulo <- which(upper.tri(matrices_distancia$dtw), arr.ind = TRUE)

pares_dtw_vs_euclidea <- data.table(
  artista_1 = artistas$artist_name[indices_triangulo[, 1]],
  artista_2 = artistas$artist_name[indices_triangulo[, 2]],
  euclidea = matrices_distancia$euclidea[indices_triangulo],
  dtw = matrices_distancia$dtw[indices_triangulo]
)

pares_dtw_vs_euclidea[, cociente_dtw_euclidea := dtw / euclidea]

# Sólo pares con distancia euclídea relevante (sobre la mediana):
# entre series casi iguales el cociente no dice nada
cat("Distribución del cociente DTW / euclídea:\n")
print(round(quantile(pares_dtw_vs_euclidea$cociente_dtw_euclidea, c(0, 0.01, 0.1, 0.5, 0.9, 1)), 3))

pares_dtw_vs_euclidea[euclidea > median(euclidea)][order(cociente_dtw_euclidea)][1:10]

In [ ]:
# ============================================================
# FIGURA: distancia euclídea vs DTW para todos los pares
# Escala de grises: más oscuro, más pares. La diagonal de
# igualdad va en blanco sobre la banda densa.
# ============================================================

grafico_euclidea_vs_dtw <- ggplot(pares_dtw_vs_euclidea, aes(x = euclidea, y = dtw)) +
  geom_bin2d(bins = 120) +
  geom_abline(slope = 1, intercept = 0, colour = "white", linetype = "dashed", linewidth = 0.7) +
  scale_fill_gradientn(
    colours = c("#D9D9D9", "#9A9A9A", "#4A4A4A", "#111111"),
    trans = "log10",
    name = "Pares de
artistas"
  ) +
  coord_equal() +
  labs(
    title = "DTW frente a la distancia euclídea",
    subtitle = paste0(
      "Cada punto es un par de artistas.
",
      "Bajo la diagonal, DTW los encuentra más parecidos ",
      "(ventana de ", VENTANA_DTW_MESES, " meses)"
    ),
    x = "Distancia euclídea",
    y = "Distancia DTW"
  ) +
  theme_minimal(base_size = 13, base_family = "LWR Mono") +
  theme(
    plot.title = element_text(family = "LWR Titulo", size = 26, colour = "#111111", margin = margin(b = 8)),
    plot.subtitle = element_text(size = 12, colour = "#4A4A4A", lineheight = 1.15, margin = margin(b = 14)),
    axis.title = element_text(face = "bold", colour = "#111111"),
    axis.text = element_text(colour = "#6B6B6B"),
    legend.title = element_text(size = 11, colour = "#4A4A4A"),
    legend.text = element_text(size = 10, colour = "#4A4A4A"),
    panel.grid.minor = element_blank(),
    panel.grid.major = element_line(colour = "#EDEDED"),
    plot.title.position = "plot",
    plot.background = element_rect(fill = "white", colour = NA),
    plot.margin = margin(18, 18, 12, 18)
  )

ggsave(
  file.path(FIGURES_DIR, "clustering_01_euclidea_vs_dtw.png"),
  grafico_euclidea_vs_dtw,
  width = 9, height = 8, dpi = 300, bg = "white",
  device = ragg::agg_png
)

# El dispositivo del notebook no conoce las tipografías registradas:
# se muestra el PNG guardado
IRdisplay::display_png(file = file.path(FIGURES_DIR, "clustering_01_euclidea_vs_dtw.png"))


In [ ]:
# ============================================================
# GUARDADO DE LAS MATRICES (archivo local, excluido de git)
# ============================================================

saveRDS(
  list(
    fecha_generacion = Sys.time(),
    archivo_panel = ARCHIVO_PANEL_PREPROCESADO,
    columna_serie = COLUMNA_SERIE,
    variables = VARIABLES_CLUSTERING,
    ventana_dtw_meses = VENTANA_DTW_MESES,
    artistas = artistas,
    matrices = matrices_distancia,
    sensibilidad = matrices_sensibilidad
  ),
  ARCHIVO_DISTANCIAS
)

cat("Matrices guardadas en", ARCHIVO_DISTANCIAS, "-",
    round(file.size(ARCHIVO_DISTANCIAS) / 1e6, 1), "MB\n")